# 第100章 不平衡分类与阈值选择

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 15 / 34 步：从模型分数走向业务评价与阈值**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 交叉验证与超参数调优  →  **本章任务：** 不平衡分类与阈值选择  →  **下一步：** 随机森林回归
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

**背景引入**：在真实业务里，"正例"总是很少见——故障告警、异常交易、用户流失往往只占样本的一小部分。如果不管它，模型很容易学会"全都预测成多数类"，准确率看着很高，却把真正要紧的问题全漏掉了。这一章就来处理这种不平衡数据：先构造一个不平衡任务看清准确率的陷阱，再用类别权重（class_weight）和阈值选择，让评价指标真正贴近业务关心的漏判与误报。


## 本章目标

学完本章，你将能够：

- **理解**：理解「不平衡分类与阈值选择」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「不平衡分类与阈值选择」的关键输出指标。
- **迁移**：能把「不平衡分类与阈值选择」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 核心概念

**背景引入**：当正类极其稀少（比如只有 1% 的客户会购买）时，“准确率”几乎失去意义——哪怕模型把所有人都判成“不买”，准确率也有 99%。真正要决策的是：在漏掉正类和多误报之间怎么权衡。类别权重或重采样改变的是训练损失，而真正一刀切的，是那个决策阈值。


- PR-AUC 的基线等于正类比例
- 类别权重改变训练损失而非数据本身
- 阈值决定最终错误成本（打个比方：像调报警器的灵敏度——线定太高，真危险来了漏报；太低，天天误报烦死你；怎么定，要看你更心疼哪种“错账”。）
- 重采样和阈值必须只依据训练/验证数据设计


## 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 构造不平衡任务 | `np.random.default_rng()`、`np.flatnonzero()`、`original.to_numpy()`、`rng.choice()` | 从乳腺癌数据中固定抽取较少正类，演示准确率为何可能误导。 | 不平衡任务只报告准确率 |
| 权重、概率与阈值 | `m.predict_proba()`、`rows.append()`、`pd.DataFrame()`、`.fit()` | 比较普通与平衡权重模型，并按验证目标选择阈值。 | 在测试集上选择阈值再报告测试性能 |
| 有限容量 Top-K | `pd.DataFrame()`、`ranked.head()`、`top.y.mean()`、`ranked.y.mean()` | 当人工复核只能覆盖 10% 样本时，直接评价最高分名单。 | 使用类别权重后仍假设概率天然校准 |


## 例 1｜构造不平衡任务

从乳腺癌数据中固定抽取较少正类，演示准确率为何可能误导。


<!-- math-foundation:chapter-100 -->
### 数学推导｜阈值连接概率与错误成本

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜阈值把概率变成行动。** $\hat y_i(t)=\mathbf1(p_i\ge t)$。

**第 2 步｜由逐样本条件累计错误。** 

$$
FP(t)=\sum_i\mathbf1(y_i=0,\hat y_i(t)=1),
\quad
FN(t)=\sum_i\mathbf1(y_i=1,\hat y_i(t)=0)
$$

**第 3 步｜把不同错误换成同一成本单位。** $Cost(t)=c_{FP}FP(t)+c_{FN}FN(t)$，在验证集候选阈值中选择成本最小者，而不是默认 0.5。

**把上面的关系收束为本章计算式：**

$$
\hat{y}_t=\mathbf{1}(p\ge t),\qquad Cost(t)=c_{FP}FP(t)+c_{FN}FN(t)
$$

**符号解释：** $t$ 是阈值，$c_{FP}$、$c_{FN}$ 是两类错误成本。

**代码对应：** 遍历多个阈值，记录 precision、recall、行动数量和业务成本。

**使用边界：** 阈值必须在验证集确定；测试集不能反复用于选择。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer(as_frame=True)
X = data.data
original = data.target
rng = np.random.default_rng(89)
positive_idx = np.flatnonzero(original.values == 0)
negative_idx = np.flatnonzero(original.values == 1)
keep = np.r_[rng.choice(positive_idx, 60, replace=False), negative_idx]
X_imb = X.iloc[keep]
y_imb = (original.iloc[keep].values == 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X_imb, y_imb, stratify=y_imb, test_size=0.3, random_state=89
)
print(
    "正类比例:",
    round(y_imb.mean(), 3),
    " 全预测负类准确率:",
    round(1 - y_test.mean(), 3),
)


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：把构造不平衡数据时抽取的正类样本数量从 `n_pos = 60` 改成 `n_pos = 100`，重新运行，观察正类比例 `new_pos_rate` 如何变化。想一想：正类变多之后，模型"全都预测成负类"所能拿到的那个最高"准确率"会上升还是下降？

只改动 `n_pos` 这一个数字即可；下面的 `# 自检` 会提示你的新正类比例是否符合直觉。


In [ ]:
try:
    # 请在下方填写代码
    # 练习：把抽取正类样本的数量从 n_pos = 60 改为 n_pos = 100，核对新的正类比例。
    import numpy as np
    import pandas as pd
    from sklearn.datasets import load_breast_cancer

    # TODO: 把 n_pos 从 60 改为 100
    n_pos = 60

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜权重、概率与阈值

比较普通与平衡权重模型，并按验证目标选择阈值。


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
)

rows = []
models = {}
for name, weight in [("普通", None), ("平衡", "balanced")]:
    m = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight=weight),
    ).fit(X_train, y_train)
    p = m.predict_proba(X_test)[:, 1]
    pred = p >= 0.5
    models[name] = (m, p)
    rows.append(
        [
            name,
            roc_auc_score(y_test, p),
            average_precision_score(y_test, p),
            precision_score(y_test, pred),
            recall_score(y_test, pred),
        ]
    )
display(
    pd.DataFrame(
        rows, columns=["模型", "ROC_AUC", "PR_AUC", "precision", "recall"]
    )
    .set_index("模型")
    .round(3)
)


## 例 3｜有限容量 Top-K

当人工复核只能覆盖 10% 样本时，直接评价最高分名单。


In [ ]:
prob = models["平衡"][1]
ranked = pd.DataFrame({"y": y_test, "prob": prob}).sort_values(
    "prob", ascending=False
)
k = max(1, int(len(ranked) * 0.1))
top = ranked.head(k)
lift = top.y.mean() / ranked.y.mean()
print(f"Top 10% 样本数：{k}")
print(f"precision={top.y.mean():.3f}，lift={lift:.2f}")
print(f"recall={top.y.sum() / ranked.y.sum():.3f}")


## 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # TODO: 在此粘贴或改写最接近的示例。
    # 记录：我改了什么？预期会发生什么？实际观察到什么？
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(
        {"修改": change_note, "预期": expected_change, "观察": observed_change}
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

_demo_X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(_demo_X, y)
model = LinearRegression().fit(_demo_X, y)
print("基线预测：", np.round(baseline.predict(_demo_X[:2]), 2))
print("模型预测：", np.round(model.predict(_demo_X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(_demo_X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(_demo_X)), 2))


### 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = _demo_X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(_demo_X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print(
    "预测变化：",
    np.round(changed_prediction[:2] - model.predict(_demo_X[:2]), 2),
)


### 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

_demo_data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in _demo_data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 不平衡任务只报告准确率
- 在测试集上选择阈值再报告测试性能
- 使用类别权重后仍假设概率天然校准
- 只追求召回率而不考虑人工容量和误报成本


## 练习与作业

1. 在平衡权重模型上寻找召回率至少 0.8 的最高阈值
2. 报告精确率
3. 输出阈值和两个指标

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 100.12 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“在平衡权重模型上寻找召回率至少 0.8 的最高阈值”。
2. **独立完成**：不复制示例代码，完成“报告精确率”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“输出阈值和两个指标”，用一两句话说明你修改了什么。

### 100.12.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 100.12.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


## 小结

在类别不平衡任务中使用 PR-AUC、类别权重和阈值选择，使模型评价与漏判、误报和业务容量一致。


### 你已经掌握

- 识别准确率陷阱
- 计算 ROC-AUC 与 PR-AUC
- 使用 class_weight
- 按目标召回率或有限容量选择阈值


### 需要注意

- 不平衡任务只报告准确率
- 在测试集上选择阈值再报告测试性能
- 使用类别权重后仍假设概率天然校准
- 只追求召回率而不考虑人工容量和误报成本


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# 完整答案：把抽取正类样本的数量从 60 改为 100
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
X = data.data
original = data.target
rng = np.random.default_rng(89)
positive_idx = np.flatnonzero(original.values == 0)
negative_idx = np.flatnonzero(original.values == 1)

n_pos = 100  # 把正类样本数量改大
keep = np.r_[rng.choice(positive_idx, n_pos, replace=False), negative_idx]
X_imb = X.iloc[keep]
y_imb = (original.iloc[keep].values == 0).astype(int)

new_pos_rate = y_imb.mean()
print("正类样本数:", int(y_imb.sum()), "  正类比例:", round(new_pos_rate, 3))


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
choices = []
for threshold in np.linspace(0.05, 0.95, 91):
    pred = prob >= threshold
    recall = recall_score(y_test, pred)
    if recall >= 0.8:
        choices.append((threshold, precision_score(y_test, pred), recall))
selected = max(choices, key=lambda x: x[0])
print("threshold, precision, recall:", tuple(round(v, 3) for v in selected))
